### Timing: Depending on the panel size, 30 min - 6 h

This panel is used for panel design and codebook generation (assign genes to cycles and coding colors).

environment: openFISH_genepanel  
First part of this program is for panel design, which can be skipped if a pre-designed panel is given;  
Second part of this program is for codebook generation. This also allows for adding genes to the blank code from existing codebook.

## Panel design

In [1]:
import pandas as pd
import scanpy as sc
import numpy as np

from scGIST import scGIST
from scGIST import test_classifier, get_priority_score_list

2026-01-16 12:58:51.865664: I tensorflow/stream_executor/platform/default/dso_loader.cc:53] Successfully opened dynamic library libcudart.so.11.0


### Load reference RNAseq data

adata needs to have been normalized

In [2]:
adata = sc.read_h5ad("/path/to/reference_scRNA_seq.h5ad")
adata

AnnData object with n_obs × n_vars = 87263 × 18384
    obs: 'cell_barcode_x', 'library_label_x', 'anatomical_division_label', 'cell_barcode_y', 'barcoded_cell_sample_label', 'library_label_y', 'feature_matrix_label', 'entity', 'brain_section_label', 'library_method', 'region_of_interest_acronym', 'donor_label', 'donor_genotype', 'donor_sex', 'dataset_label', 'x', 'y', 'cluster_alias', 'neurotransmitter', 'class', 'subclass', 'supertype', 'cluster', 'neurotransmitter_color', 'class_color', 'subclass_color', 'supertype_color', 'cluster_color', 'region_of_interest_order', 'region_of_interest_color', 'CellType', 'batch', 'n_genes_by_counts', 'total_counts', 'n_counts'
    var: 'gene_symbol_y', 'name', 'mapped_ncbi_identifier', 'comment', 'gene_identifier', 'mt_gene', 'Gm_gene', 'Rik_gene', 'Rpl_gene', 'Rps_gene', 'AW_gene', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'mean', 'std'
    uns: 'CellType_colors', 'class_colors', 'log1p'
    obsm: 'X_u

____________________________________________________
Optional: **Input pre-selected genes**

In [21]:
# You can define a set of genes to be considered, these pre-selected genes will be considered with priority when design panel.
selected_genes = [
    "Cck", "Cbln1", "Dner", "Synpr", "Nfib",
    "Satb2", "Bcl11b", "Shox2", "Zfpm2", "Otp",
    "Nptxr", "Pdzrn3", "Shisa6", "Sst", "Tcf7l2",
    "Cplx2", "Ntng1", "Mpped1", "Atp2b4", "C1ql3",
    "Slc24a3", "Zeb2", "Slc17a6", "Lmo3", "Gad2",
    "Nxph3", "Fgf13", "Nr2f2", "Hap1", "Etv1", "Pvalb",
    "Meis2", "Rspo2", "Slc1a3", "Htr2c", "Slc32a1", "Nrn1",
    "Slit3", "Ptprt", "Vxn", "Prox1", "Lamp5", "Vip", "Reln",
    "Dpp6", "Slc17a7", "Gad1", "Drd1", "Drd2", "Trh"
]

df = pd.DataFrame({"gene_name": selected_genes})
df["priority"] = 1
df = get_priority_score_list(adata, df)

___________________________________
Optional: **Exclude highly expressed genes (99% percentile)**

In [57]:
tmp_df = adata.to_df()
tmp_df = tmp_df.sum()

tmp_df = tmp_df[tmp_df <= np.percentile(tmp_df, 99)]
adata = adata[:, tmp_df.index].copy()

# You can falso iltered genes according to your own need (mito, ribo, lnc, lowly expressed .et.al.). Write your own code

#### 
________

### Design the panel using scGIST

In [32]:
label_column = "subclass" # Define the reference cluster label. (i.e. class, subclass, cell_type et.al.)
panel_size = 110 # Panel size
iter_num = 10 # scGIST results differ slightly between rounds, try multiple times and select a optimal one

panel_results = []
n_classes = adata.obs[label_column].unique().size
n_genes = adata.X.shape[1]

In [34]:
for i in range(iter_num):

    gist = scGIST()

    try:
        priority_scores = df
        p_criteria = True
    except NameError:
        priority_scores = None
        p_criteria = False

    
    gist.create_model(n_genes, n_classes, panel_size=panel_size, alpha=1.5, strict = True, priority_scores=priority_scores, beta = 0.4)
    """
    n_features: no. of features/cells
    n_classes: no. of classes/clusters/labels
    panel_size: Total no. of features to be taken
    priority_scores: List of features we are interested in
    pairs: Pairs of genes which should be included or excluded together
    alpha: strictness of the panel size
    beta: priority coefficient
    gamma: likeliness to take pairs together
    strict: when True, the model will select exactly the same amount of genes specified in the panel size; when False, the model will select less than or equal to the amount of genes specified.
    """
    gist.compile_model()
    gist.train_model(adata, label_column, verbose=0, epochs=300)

    markers_indices = gist.get_markers_indices()
    accuracy, f1 = test_classifier(adata, label_column, markers=markers_indices)
    panel = gist.get_markers_names(adata, plot_weights=False)
    panel_results.append(panel)

    if p_criteria:
        tmp = [x for x in panel if x in selected_genes]
    else:
        tmp = 0
        
    print(f"Panel No.{i}: The panel accuracy is {accuracy}, f1score is {f1}, {len(tmp)} prioritized genes.")

Panel No.0: The panel accuracy is 96.30619962565416, f1score is 0.9645725618550257, 36 prioritized genes.
Panel No.1: The panel accuracy is 96.53157110661216, f1score is 0.9668073938042614, 36 prioritized genes.


**Select the best panel and save it**

In [35]:
panel = panel_results[1] # Select the best iteration
np.save("./best_panel.npy", panel)

## Cycle and RO assign

### Imputation

We need to imputate the data to enable a more reasonable correlation indices calculation

In [36]:
import magic

magic_op = magic.MAGIC(solver='approximate')

In [37]:
adata_magic = magic_op.fit_transform(adata.to_df(), genes="all_genes") # Choose normalized data to imputate
adata.layers["MAGIC"] = adata_magic

# Save the adata
adata.write_h5ad("Test_maigc.h5ad")

Calculating MAGIC...
  Running MAGIC on 87263 cells and 17658 genes.
  Calculating graph and diffusion operator...
    Calculating PCA...
    Calculated PCA in 19.75 seconds.
    Calculating KNN search...
    Calculated KNN search in 523.11 seconds.
    Calculating affinities...
    Calculated affinities in 498.38 seconds.
  Calculated graph and diffusion operator in 1041.92 seconds.
  Calculating imputation...
  Calculated imputation in 0.32 seconds.
Calculated MAGIC in 1046.96 seconds.


### Define GeneticAlgorithm

In [54]:
import numpy as np
import scanpy as sc
import random
import os
from scipy.stats import spearmanr
from sklearn.preprocessing import MinMaxScaler
from joblib import Parallel, delayed

class GeneticChoose():
    def __init__(self, 
                 panel, # input panel, list or numpy.ndarray
                 ref_path, # Path to  imputated h5ad file
                 input_order = {'R1': slice(0, 10, 1),
                                 'R2': slice(10, 20, 1),
                                 'R3': slice(20, 30, 1),
                                 'R4': slice(30, 40, 1),
                                 'R5': slice(40, 50, 1),
                                 'R6': slice(50, 60, 1),
                                 'R7': slice(60, 70, 1),
                                 'R8': slice(70, 80, 1),
                                 'R9': slice(80, 90, 1),
                                 'R10': slice(90, 100, 1),
                                 'R11': slice(100, 110, 1)},

                 magic_layer = 'MAGIC', threads = 128, pop_size = 127, num_generations = 300):
        
        self.pop_size = pop_size
        self.num_generations = num_generations
        self.threads = threads
        
        adata = sc.read_h5ad(ref_path)
        self.MAGIC_df = adata.to_df(layer = magic_layer)

        self.input_order = input_order
        self.panel = np.array(panel)
        
    # Fitness function
    def Comb_spearmanr(self, genes_perm):

        ORDER = {key: genes_perm[value] for key,value in self.input_order.items()}
            
        comb_R = 0
        
        for tmp_comb in ORDER.values():
            
            spearmanr_matrix = spearmanr(self.MAGIC_df.loc[:, tmp_comb]).correlation
            spearmanr_mean = spearmanr_matrix[np.tril_indices(spearmanr_matrix.shape[0], k=-1)].mean()
            
            comb_R += spearmanr_mean

        return comb_R

    # Individual Selection
    def selection(self, population):
        scaler = MinMaxScaler(feature_range=(1, 10)) # scale the data to enlarge the differences
        population_spearmanr = Parallel(n_jobs=self.threads, backend='loky')(delayed(self.Comb_spearmanr)(x) for x in population)
        # print(f"Present minimum spearmanr aggregation: {min(population_spearmanr)}")
        scaled_spearmanr = scaler.fit_transform(np.array(population_spearmanr).reshape(-1,1))
        fitness = [1 / x for x in scaled_spearmanr.flatten()]
        idx = np.random.choice(len(population), size=len(population), p=fitness/np.sum(fitness))
        return [population[i] for i in idx]

    # CX
    def crossover(self, parent1, parent2):

        child = [-1] * len(parent1)

        idx = np.random.choice(len(parent1), 1)[0]

        while child[idx] == -1:
            child[idx] = parent1[idx]
            idx = parent1.index(parent2[idx])

        for i in range(len(child)):
            if child[i] == -1:
                child[i] = parent2[i]

        return child

    # mutation
    def mutation(self, individual):
        idx1, idx2 = np.random.choice(len(individual), size=2, replace=False)
        tmp1 = individual[idx1]
        tmp2 = individual[idx2]
        individual[idx1] = tmp2
        individual[idx2] = tmp1
        return individual

    # Genetic algorithm main function
    def genetic_algorithm(self):

        genes = self.panel
        population = [list(np.random.permutation(genes)) for _ in range(self.pop_size)]

        prod_list = []
        for ig in range(self.num_generations):
            
            if ig%10 == 0:
                print(f"Generation {ig}:")
            
            new_population = []

            while len(new_population) < self.pop_size:
                parent1_idx, parent2_idx = np.random.choice(len(population), size=2, replace=False)
                parent1 = list(population[parent1_idx])
                parent2 = list(population[parent2_idx])
                child = self.crossover(parent1, parent2)
                if np.random.rand() < 0.5:  # 50% probability mutate
                    child = self.mutation(child)
                new_population.append(np.array(child))

            population = self.selection(new_population)

        population_spearmanr = Parallel(n_jobs=self.threads, backend='loky')(delayed(self.Comb_spearmanr)(x) for x in population)
        best_index = population_spearmanr.index(min(population_spearmanr))
        best_perm = population[best_index]
        
        return best_perm

### Define the parameters for Cycle&Channel arrangement

In [55]:
input_panel_path = "best_panel.npy"
output_panel_path = 'best_panel.csv' # Final results will be output here
ref_path = 'Test_maigc.h5ad' # Path to  imputated h5ad file

# Define expected arrangement of the Channel and Cycle
# The slots number of perm_comb has to be same with the panel size. If you have some slots occupied already, remove them from the perm_comb
# Below assume all slots for 110 genes were used
perm_comb = {
            'R1': ['AF488_AF546', 'AF488_AF594', 'AF488_AF647', 'AF488_AF750', 'AF546_AF594', 'AF546_AF647', 'AF546_AF750', 'AF594_AF647', 'AF594_AF750', 'AF647_AF750'],
            'R2': ['AF488_AF546', 'AF488_AF594', 'AF488_AF647', 'AF488_AF750', 'AF546_AF594', 'AF546_AF647', 'AF546_AF750', 'AF594_AF647', 'AF594_AF750', 'AF647_AF750'],
            'R3': ['AF488_AF546', 'AF488_AF594', 'AF488_AF647', 'AF488_AF750', 'AF546_AF594', 'AF546_AF647', 'AF546_AF750', 'AF594_AF647', 'AF594_AF750', 'AF647_AF750'],
            'R4': ['AF488_AF546', 'AF488_AF594', 'AF488_AF647', 'AF488_AF750', 'AF546_AF594', 'AF546_AF647', 'AF546_AF750', 'AF594_AF647', 'AF594_AF750', 'AF647_AF750'],
            'R5': ['AF488_AF546', 'AF488_AF594', 'AF488_AF647', 'AF488_AF750', 'AF546_AF594', 'AF546_AF647', 'AF546_AF750', 'AF594_AF647', 'AF594_AF750', 'AF647_AF750'],
            'R6': ['AF488_AF546', 'AF488_AF594', 'AF488_AF647', 'AF488_AF750', 'AF546_AF594', 'AF546_AF647', 'AF546_AF750', 'AF594_AF647', 'AF594_AF750', 'AF647_AF750'],
            'R7': ['AF488_AF546', 'AF488_AF594', 'AF488_AF647', 'AF488_AF750', 'AF546_AF594', 'AF546_AF647', 'AF546_AF750', 'AF594_AF647', 'AF594_AF750', 'AF647_AF750'],
            'R8': ['AF488_AF546', 'AF488_AF594', 'AF488_AF647', 'AF488_AF750', 'AF546_AF594', 'AF546_AF647', 'AF546_AF750', 'AF594_AF647', 'AF594_AF750', 'AF647_AF750'],
            'R9': ['AF488_AF546', 'AF488_AF594', 'AF488_AF647', 'AF488_AF750', 'AF546_AF594', 'AF546_AF647', 'AF546_AF750', 'AF594_AF647', 'AF594_AF750', 'AF647_AF750'],
            'R10': ['AF488_AF546', 'AF488_AF594', 'AF488_AF647', 'AF488_AF750', 'AF546_AF594', 'AF546_AF647', 'AF546_AF750', 'AF594_AF647', 'AF594_AF750', 'AF647_AF750'],
            'R11': ['AF488_AF546', 'AF488_AF594', 'AF488_AF647', 'AF488_AF750', 'AF546_AF594', 'AF546_AF647', 'AF546_AF750', 'AF594_AF647', 'AF594_AF750', 'AF647_AF750'],
        }

### Run the Cycle&Channel arragement algorithm

In [56]:
panel = np.load(input_panel_path)

perm_comb_length = {k:len(x) for k,x in perm_comb.items()}
assert sum(perm_comb_length.values()) == len(panel), f'Panel size({len(panel)}) and perm_comb({sum(perm_comb_length.values())}) must have same length'

CYCLES = {}
i = 0
for cycle, length in perm_comb_length.items():
    assert length <= 10, f"{cycle} has component length({length}) larger than 10, check the iinput"
    CYCLES[cycle] = slice(i, length + i, 1)
    i += length

ga = GeneticChoose(panel, # input panel, list or numpy.ndarray
                                 ref_path = ref_path, 
                                 input_order = CYCLES,
                                 magic_layer = 'MAGIC', threads = 128, pop_size = 127, num_generations = 300) # define the threads and num_generations

import time

start_time = time.time()

best_perm = ga.genetic_algorithm()

trans_dict = {
    'AF488': {
        'AF546': 0, 'AF594': 1, 'AF647': 2, 'AF750': 3
    },
    'AF546': {
        'AF488': 0, 'AF594': 4, 'AF647': 5, 'AF750': 6
    },
    'AF594': {
        'AF546': 4, 'AF488': 1, 'AF647': 7, 'AF750': 8
    },
    'AF647': {
        'AF546': 5, 'AF594': 7, 'AF488': 2, 'AF750': 9
    },
    'AF750': {
        'AF546': 6, 'AF594': 8, 'AF647': 9, 'AF488': 3
    }
}

BEST_ORDER = {key: best_perm[value] for key,value in CYCLES.items()}

Add_df = []

for key, value in perm_comb.items():
    CHANNELS = {}
    for cc in value:
        c1, c2 = cc.split("_")
        try:
            CHANNELS[c1].append(trans_dict[c1][c2])
        except KeyError:
            CHANNELS[c1] = [trans_dict[c1][c2]]

        try:
            CHANNELS[c2].append(trans_dict[c2][c1])
        except KeyError:
            CHANNELS[c2] = [trans_dict[c2][c1]]

    
    ga = GeneticChoose(best_perm[CYCLES[key]], # input panel, list or numpy.ndarray
                                 ref_path = 'Test_maigc.h5ad', # Path to  imputated h5ad file
                                 input_order = CHANNELS,
                                 magic_layer = 'MAGIC', threads = 128, pop_size = 127, num_generations = 100) # define the threads and num_generations
    
    best_perm_channel = ga.genetic_algorithm()

    df = pd.DataFrame({
        'Genes': best_perm_channel,
        'RO1': [cc.split("_")[0] for cc in value],
        'RO2': [cc.split("_")[1] for cc in value],
        'Round': key
    })

    Add_df.append(df)

Final_df = pd.concat(Add_df)
Final_df.to_csv(output_panel_path, index = False)

end_time = time.time()
elapsed_time = end_time - start_time

print(f"Run time is：{elapsed_time/3600}hours.")

Generation 0:
Present minimum spearmanr aggregation: 0.30363530205295136
Present minimum spearmanr aggregation: 0.20833871006597685
Generation 0:
Present minimum spearmanr aggregation: -0.008379649851904808
Present minimum spearmanr aggregation: 0.005028872162291348
Generation 0:
Present minimum spearmanr aggregation: -0.11961129243354147
Present minimum spearmanr aggregation: -0.16859878762157313
Generation 0:
Present minimum spearmanr aggregation: -0.2064330357378804
Present minimum spearmanr aggregation: -0.2288801077461089
Generation 0:
Present minimum spearmanr aggregation: 0.011815523864893762
Present minimum spearmanr aggregation: -0.018012188269877667
Generation 0:
Present minimum spearmanr aggregation: -0.013055727574472542
Present minimum spearmanr aggregation: -0.0007281946805766659
Generation 0:
Present minimum spearmanr aggregation: -0.38685434154630494
Present minimum spearmanr aggregation: -0.41926701470182054
Generation 0:
Present minimum spearmanr aggregation: -0.10205